In [4]:
import numpy as np

# -------------------------
# precision control
# -------------------------
# Switch between np.float64 and np.longdouble here.
DT = np.longdouble


def asDT(x):
    return np.asarray(x, dtype=DT)

"""
CEF Toy Model + Finite-Difference Verifier (more realistic)
===========================================================

PHASE MODEL
-----------
Two sublattices:
  s=0 : {A, B}     with site ratio v0 = 1
  s=1 : {X, Y}     with site ratio v1 = 2   (two sites on sublattice 1)

Because v1=2, the possible end-member "configurations" on sublattice 1 are:
  X2, XY, Y2
So total end members = 2 * 3 = 6:
  0: AX2
  1: AXY
  2: AY2
  3: BX2
  4: BXY
  5: BY2

VARIABLES
---------
End-member moles: n_i (i=0..5)
Site fractions:   y_A^(0), y_B^(0), y_X^(1), y_Y^(1)

SITE FRACTION MAPPING (CEF-consistent)
--------------------------------------
For sublattice s, total "site moles" are:
  N^(s) = v_s * n_total
Constituent site moles:
  N_j^(s) = Σ_i n_i * ν_{i,s,j}
where ν_{i,s,j} is how many sites of constituent j appear in end member i on sublattice s.
Then:
  y_j^(s) = N_j^(s) / N^(s)

ENERGY MODEL
------------
Molar Gibbs energy g(y) = g_ref + g_id + g_ex
Extensive Gibbs energy   G(n) = n_total * g( y(n) )

Reference term:
  g_ref = Σ_i g0_i * y_(s=0) * yX^(nuX) * yY^(nuY)

Ideal term:
  g_id = RT Σ_s v_s Σ_j y_j^(s) ln y_j^(s)

Excess term (3 branches depending on sublattice-1 configuration):
  Branch X2: M = yX^2
  Branch XY: M = yX*yY
  Branch Y2: M = yY^2
Each branch:
  phi = M * yA*yB * L(Δ), where Δ = yA - yB
Total g_ex = phi_X2 + phi_XY + phi_Y2

VERIFICATION
------------
1) Build analytic site-fraction Hessian B = ∂²g/∂y∂y (size 4x4).
2) Contract to end-member mole Hessian H (size 6x6).
3) Independently compute FD Hessian in n-space using central differences:
     mu_i ≈ [G(n+h e_i) - G(n-h e_i)]/(2h)
     H(:,m) ≈ [mu(n+h e_m) - mu(n-h e_m)]/(2h)
4) Compare analytic vs FD (relative norm error) across step sizes.
"""

# ---------- constants ----------
R = DT("8.314")
T = DT("1000.0")  # K

# ---------- site ratios v_s ----------
v = np.array([DT("1.0"), DT("2.0")], dtype=DT)   # v0=1 (A/B), v1=2 (X/Y)

# ---------- sublattice constituents ----------
sublattices = [
    ["A", "B"],  # s=0
    ["X", "Y"],  # s=1
]

# ---------- end members ----------
end_names = ["AX2", "AXY", "AY2", "BX2", "BXY", "BY2"]
n_em = len(end_names)

# ν_{i,s,j}: number of sites of constituent j on sublattice s in end member i
nu = {i: {0: {"A": 0, "B": 0}, 1: {"X": 0, "Y": 0}} for i in range(n_em)}

# AX2
nu[0][0]["A"] = 1; nu[0][1]["X"] = 2
# AXY
nu[1][0]["A"] = 1; nu[1][1]["X"] = 1; nu[1][1]["Y"] = 1
# AY2
nu[2][0]["A"] = 1; nu[2][1]["Y"] = 2
# BX2
nu[3][0]["B"] = 1; nu[3][1]["X"] = 2
# BXY
nu[4][0]["B"] = 1; nu[4][1]["X"] = 1; nu[4][1]["Y"] = 1
# BY2
nu[5][0]["B"] = 1; nu[5][1]["Y"] = 2

# Reference end-member energies g0_i (J/mol) (toy numbers)
g0 = np.array([DT("-20000"), DT("-19000"), DT("-18000"), DT("-15000"), DT("-14000"), DT("-13000")], dtype=DT)

# ---------- RK-like polynomials for 3 branches ----------
L_X2 = [DT("2000.0"), DT("-500.0"), DT("100.0")]
L_XY = [DT("1600.0"), DT("100.0"), DT("50.0")]
L_Y2 = [DT("1500.0"), DT("200.0"), DT("80.0")]

def L_poly(delta, coeffs):
    out = DT("0.0")
    p = DT("1.0")
    for a in coeffs:
        out += a * p
        p *= delta
    return out

def L_prime(delta, coeffs):
    out = DT("0.0")
    for vv in range(1, len(coeffs)):
        out += vv * coeffs[vv] * (delta ** (vv - 1))
    return out

def L_second(delta, coeffs):
    out = DT("0.0")
    for vv in range(2, len(coeffs)):
        out += vv * (vv - 1) * coeffs[vv] * (delta ** (vv - 2))
    return out

# ---------- flatten y variables: (s,name) -> alpha index ----------
alpha_list = []
alpha_index = {}
for s, consts in enumerate(sublattices):
    for name in consts:
        alpha_index[(s, name)] = len(alpha_list)
        alpha_list.append((s, name))
n_alpha = len(alpha_list)  # should be 4

# ---------- y(n) mapping ----------
def y_from_n(n):
    """
    y_j^(s) = N_j^(s) / N^(s)
    N_j^(s) = Σ_i n_i * ν_{i,s,j}
    N^(s)   = v_s * n_total
    """
    n = asDT(n)
    n_tot = np.sum(n, dtype=DT)
    if n_tot <= 0 or np.any(n <= 0):
        return None

    y = np.zeros(n_alpha, dtype=DT)
    for s, consts in enumerate(sublattices):
        Ns = v[s] * n_tot
        for name in consts:
            Nj = sum(n[i] * nu[i][s][name] for i in range(n_em))
            y[alpha_index[(s, name)]] = Nj / Ns
    return y

# ---------- g(y) pieces ----------
def g_ref(y):
    """
    g_ref = Σ_i g0_i * (yA or yB) * yX^(nuX) * yY^(nuY)
    (Correctly uses powers because sublattice 1 has 2 sites.)
    """
    yA = y[alpha_index[(0, "A")]]
    yB = y[alpha_index[(0, "B")]]
    yX = y[alpha_index[(1, "X")]]
    yY = y[alpha_index[(1, "Y")]]

    val = DT("0.0")
    for i in range(n_em):
        s0_factor = yA if nu[i][0]["A"] == 1 else yB
        nuX = nu[i][1]["X"]
        nuY = nu[i][1]["Y"]
        prod = s0_factor * (yX ** nuX) * (yY ** nuY)
        val += g0[i] * prod
    return val

def g_id(y):
    """
    g_id = RT Σ_s v_s Σ_j y ln y
    """
    val = DT("0.0")
    for s, consts in enumerate(sublattices):
        for name in consts:
            yy = y[alpha_index[(s, name)]]
            val += v[s] * yy * np.log(yy)
    return R * T * val

def g_ex(y):
    """
    g_ex = Σ_branches M(yX,yY) * yA*yB * L(Δ)
    where Δ = yA - yB and M is:
      X2: yX^2
      XY: yX*yY
      Y2: yY^2
    """
    yA = y[alpha_index[(0, "A")]]
    yB = y[alpha_index[(0, "B")]]
    yX = y[alpha_index[(1, "X")]]
    yY = y[alpha_index[(1, "Y")]]
    delta = yA - yB

    F = yA * yB
    return (
        (yX ** 2) * F * L_poly(delta, L_X2) +
        (yX * yY) * F * L_poly(delta, L_XY) +
        (yY ** 2) * F * L_poly(delta, L_Y2)
    )

def g_total(y):
    return g_ref(y) + g_id(y) + g_ex(y)

def G_total(n):
    """
    G(n) = n_total * g(y(n))
    """
    n = asDT(n)
    y = y_from_n(n)
    if y is None:
        return np.nan
    return np.sum(n, dtype=DT) * g_total(y)

# ---------- Analytic B = ∂²g/∂y∂y ----------
def B_id(y):
    """
    For g_id = RT Σ_s v_s Σ_j y ln y:
      ∂²/∂y^2 = RT * v_s / y   (diagonal only)
    """
    B = np.zeros((n_alpha, n_alpha), dtype=DT)
    for a, (s, name) in enumerate(alpha_list):
        B[a, a] += R * T * v[s] / y[a]
    return B

def B_ref(y):
    """
    Analytic site-fraction Hessian for g_ref.
    This term is NOT just cross-sublattice now because sublattice 1 enters with powers (X2, XY, Y2).
    """
    B = np.zeros((n_alpha, n_alpha), dtype=DT)

    yA = y[alpha_index[(0, "A")]]
    yB = y[alpha_index[(0, "B")]]
    yX = y[alpha_index[(1, "X")]]
    yY = y[alpha_index[(1, "Y")]]

    aA = alpha_index[(0, "A")]
    aB = alpha_index[(0, "B")]
    aX = alpha_index[(1, "X")]
    aY = alpha_index[(1, "Y")]

    for i in range(n_em):
        g = g0[i]
        # which sublattice-0 factor?
        s0_is_A = (nu[i][0]["A"] == 1)
        a0 = aA if s0_is_A else aB
        y0 = yA if s0_is_A else yB

        nuX = nu[i][1]["X"]
        nuY = nu[i][1]["Y"]

        # f = g * y0 * yX^nuX * yY^nuY
        # cross with y0 and X/Y:
        if nuX > 0:
            val = g * (nuX * (yX ** (nuX - 1))) * (yY ** nuY)
            B[a0, aX] += val; B[aX, a0] += val
        if nuY > 0:
            val = g * (yX ** nuX) * (nuY * (yY ** (nuY - 1)))
            B[a0, aY] += val; B[aY, a0] += val

        # second derivatives on X,X and Y,Y and X,Y:
        if nuX >= 2:
            val = g * y0 * (nuX * (nuX - 1) * (yX ** (nuX - 2))) * (yY ** nuY)
            B[aX, aX] += val
        if nuY >= 2:
            val = g * y0 * (yX ** nuX) * (nuY * (nuY - 1) * (yY ** (nuY - 2)))
            B[aY, aY] += val
        if (nuX > 0) and (nuY > 0):
            val = g * y0 * (nuX * (yX ** (nuX - 1))) * (nuY * (yY ** (nuY - 1)))
            B[aX, aY] += val; B[aY, aX] += val

        # no (A,A) or (B,B) second derivatives: linear in sublattice-0 factor

    return B

def B_ex(y):
    """
    For each branch:
      phi = M(yX,yY) * F(yA,yB)
      F = yA*yB*L(Δ), Δ=yA-yB
    We use the general product rule:
      ∂²phi = M * ∂²F  +  (∂M)*(∂F)  +  (∂²M)*F
    """
    B = np.zeros((n_alpha, n_alpha), dtype=DT)

    aA = alpha_index[(0, "A")]
    aB = alpha_index[(0, "B")]
    aX = alpha_index[(1, "X")]
    aY = alpha_index[(1, "Y")]

    yA = y[aA]; yB = y[aB]; yX = y[aX]; yY = y[aY]
    delta = yA - yB

    def add_branch(M, dMdX, dMdY, d2XX, d2XY, d2YY, coeffs):
        L   = L_poly(delta,  coeffs)
        Lp  = L_prime(delta, coeffs)
        Lpp = L_second(delta, coeffs)

        # Define F and its derivatives wrt yA (a) and yB (b)
        F   = yA * yB * L
        Fa  = yB * L + yA * yB * Lp
        Fb  = yA * L - yA * yB * Lp
        Faa = DT("2.0") * yB * Lp + yA * yB * Lpp
        Fbb = -DT("2.0") * yA * Lp + yA * yB * Lpp
        Fab = L + (yA - yB) * Lp - yA * yB * Lpp

        # mixing-mixing block
        B[aA, aA] += M * Faa
        B[aB, aB] += M * Fbb
        B[aA, aB] += M * Fab
        B[aB, aA] += M * Fab

        # mixing-prefactor cross terms
        B[aA, aX] += dMdX * Fa; B[aX, aA] += dMdX * Fa
        B[aA, aY] += dMdY * Fa; B[aY, aA] += dMdY * Fa
        B[aB, aX] += dMdX * Fb; B[aX, aB] += dMdX * Fb
        B[aB, aY] += dMdY * Fb; B[aY, aB] += dMdY * Fb

        # prefactor-prefactor block
        B[aX, aX] += d2XX * F
        B[aX, aY] += d2XY * F; B[aY, aX] += d2XY * F
        B[aY, aY] += d2YY * F

    # Branch X2: M=yX^2
    add_branch(
        M=yX**2,
        dMdX=DT("2.0") * yX, dMdY=DT("0.0"),
        d2XX=DT("2.0"), d2XY=DT("0.0"), d2YY=DT("0.0"),
        coeffs=L_X2
    )
    # Branch XY: M=yX*yY
    add_branch(
        M=yX*yY,
        dMdX=yY, dMdY=yX,
        d2XX=DT("0.0"), d2XY=DT("1.0"), d2YY=DT("0.0"),
        coeffs=L_XY
    )
    # Branch Y2: M=yY^2
    add_branch(
        M=yY**2,
        dMdX=DT("0.0"), dMdY=DT("2.0") * yY,
        d2XX=DT("0.0"), d2XY=DT("0.0"), d2YY=DT("2.0"),
        coeffs=L_Y2
    )

    return B

def B_total(y):
    return B_ref(y) + B_id(y) + B_ex(y)

# ---------- Contract B -> end-member mole Hessian H ----------
def H_analytic(n):
    """
    Uses the compact contraction:
      H = (1/n_total) * U * B * U^T
    where U[i,alpha] = dbar_i(alpha) - y_alpha
    and dbar_i(alpha) = ν_{i,s,j} / v_s  (fraction of sublattice-s sites in end member i that are j)

    This is the correct generalization when sublattices have v_s > 1.
    """
    n = asDT(n)
    n_tot = np.sum(n, dtype=DT)
    y = y_from_n(n)
    B = B_total(y)

    U = np.zeros((n_em, n_alpha), dtype=DT)
    for i in range(n_em):
        for a, (s, name) in enumerate(alpha_list):
            dbar = nu[i][s][name] / v[s]
            U[i, a] = dbar - y[a]

    return (U @ B @ U.T) / n_tot

# ---------- Finite differences in n-space ----------
def mu_fd(n, h):
    """
    μ_i ≈ [G(n+h e_i) - G(n-h e_i)] / (2h)
    """
    n = asDT(n)
    h = DT(h)
    mu = np.zeros(n_em, dtype=DT)
    for i in range(n_em):
        e = np.zeros(n_em, dtype=DT); e[i] = DT("1.0")
        mu[i] = (G_total(n + h * e) - G_total(n - h * e)) / (DT("2.0") * h)
    return mu

def H_fd(n0, h):
    """
    H(:,m) ≈ [ μ(n+h e_m) - μ(n-h e_m) ] / (2h)
    """
    n0 = asDT(n0)
    h = DT(h)
    H = np.zeros((n_em, n_em), dtype=DT)
    for m in range(n_em):
        e = np.zeros(n_em, dtype=DT); e[m] = DT("1.0")
        mup = mu_fd(n0 + h * e, h)
        mum = mu_fd(n0 - h * e, h)
        H[:, m] = (mup - mum) / (DT("2.0") * h)
    return H

def fro_norm(M):
    M = asDT(M)
    return np.sqrt(np.sum(M * M, dtype=DT))

# ---------- Run verifier ----------
n0 = np.array([DT("0.40"), DT("0.30"), DT("0.20"), DT("0.15"), DT("0.10"), DT("0.05")], dtype=DT)  # AX2,AXY,AY2,BX2,BXY,BY2
Ha = H_analytic(n0)

print("Site ratios v =", v)
print("Endmembers =", end_names)
print("Base G(n0) =", G_total(n0))
# print("\nAnalytic H shape:", Ha.shape)

# print("\nStep-size sweep (rel_err should drop as h shrinks, then rise if h is too tiny):")
for h in [DT("1e-2"), DT("5e-3"), DT("1e-3"), DT("5e-4"), DT("1e-4"), DT("5e-5"), DT("1e-5"), DT("5e-6"), DT("1e-6")]:
    Hnum = H_fd(n0, h)
    rel = fro_norm(Hnum - Ha) / max(DT("1.0"), fro_norm(Ha))
    sym = fro_norm(Hnum - Hnum.T) / max(DT("1.0"), fro_norm(Hnum))
    print(f"h={float(h):7.1e}  rel_err={float(rel):9.3e}") #  symmetry_err={float(sym):9.3e}

Site ratios v = [1. 2.]
Endmembers = ['AX2', 'AXY', 'AY2', 'BX2', 'BXY', 'BY2']
Base G(n0) = -35123.751429429537982
h=1.0e-02  rel_err=8.805e-04
h=5.0e-03  rel_err=2.198e-04
h=1.0e-03  rel_err=8.790e-06
h=5.0e-04  rel_err=2.197e-06
h=1.0e-04  rel_err=8.790e-08
h=5.0e-05  rel_err=2.199e-08
h=1.0e-05  rel_err=1.008e-09
h=5.0e-06  rel_err=3.653e-09
h=1.0e-06  rel_err=8.605e-08
